# Carga base limpia a Supabase

Carga los CSV de `data/raw`, aplica limpieza y escribe las tablas `raw_*` en Supabase con `UPSERT`. La utilidad de conexion bloquea `localhost` para evitar cargas accidentales a Docker.

In [ ]:
import sys
import os
import importlib
from pathlib import Path

import pandas as pd

current_dir = Path(os.getcwd())
ROOT_DIR = current_dir if (current_dir / "src").exists() else current_dir.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import src.config as _cfg_module
importlib.reload(_cfg_module)
from src.config import settings
from src.db_utils import create_supabase_engine, get_database_host, table_counts, upsert_dataframe
from src.ingesta import DataIngestor
from src.procesamiento import DataCleaner

engine = create_supabase_engine(settings.DATABASE_URL)
print(f"Base de datos activa: {get_database_host(settings.DATABASE_URL)}")

In [ ]:
ingestor = DataIngestor(raw_dir=ROOT_DIR / "data" / "raw")
dfs = ingestor.load_all()

cleaner = DataCleaner()
df_clientes = cleaner.clean_clientes(dfs["clientes"])
df_creditos = cleaner.clean_creditos(dfs["creditos"])
df_pagos = cleaner.clean_pagos(dfs["pagos"])
df_eventos = dfs["eventos"].copy()

raw_loads = [
    ("raw_clientes", df_clientes, ["cliente_id"]),
    ("raw_creditos", df_creditos, ["credito_id"]),
    ("raw_pagos", df_pagos, ["pago_id"]),
    ("raw_eventos_app", df_eventos, ["evento_id"]),
]

for table_name, dataframe, keys in raw_loads:
    affected = upsert_dataframe(dataframe, engine, table_name, keys)
    print(f"{table_name}: {len(dataframe)} filas fuente, {affected} filas insertadas/actualizadas")

raw_counts = table_counts(engine, [table for table, _, _ in raw_loads])
display(pd.DataFrame(raw_counts.items(), columns=["tabla", "filas_en_supabase"]))